Given an array of integers nums, return the length of the longest consecutive sequence of elements that can be formed.

A consecutive sequence is a sequence of elements in which each element is exactly 1 greater than the previous element. The elements do not have to be consecutive in the original array.

You must write an algorithm that runs in O(n) time.

Example 1:

Input: nums = [2,20,4,10,3,4,5]

Output: 4

Explanation: The longest consecutive sequence is [2, 3, 4, 5].

Example 2:

Input: nums = [0,3,2,5,4,6,1,1]

Output: 7

Constraints:

0 <= nums.length <= 1000

-10^9 <= nums[i] <= 10^9

In [ ]:
from typing import List

# Brute Force

Intuition

A consecutive sequence grows by checking whether the next number (num + 1, num + 2, …) exists in the set.

The brute-force approach simply starts from every number in the list and tries to extend a consecutive streak as far as possible.

For each number, we repeatedly check if the next number exists, increasing the streak length until the sequence breaks.

Even though this method works, it does unnecessary repeated work because many sequences get recomputed multiple times.

Algorithm

Convert the input list to a set for O(1) lookups.

Initialize res to store the maximum streak length.

For each number num in the original list:

Start a new streak count at 0.

Set curr = num.

While curr exists in the set:

Increase the streak count.

Move to the next number (curr += 1).

Update res with the longest streak found so far.

Return res after checking all numbers.

In [ ]:
# Brute Force
# Time Complexity: O(n^2)
# Space Complexity: O(n)
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        res = 0
        store = set(nums)

        for num in nums:
            streak, curr = 0, num
            while curr in store:
                streak += 1
                curr += 1
            res = max(res, streak)
        return res

# Sorting

Intuition

If we sort the numbers first, then all consecutive values will appear next to each other.

This makes it easy to walk through the sorted list and count how long each consecutive sequence is.

We simply move forward while the current number matches the expected next value in the sequence.

Duplicates don’t affect the result—they are just skipped—while gaps reset the streak count.

This approach is simpler and more organized than the brute force method because sorting places all potential sequences in order.

Algorithm

If the input list is empty, return 0.

Sort the array in non-decreasing order.

Initialize:

res to track the longest streak, curr as the first number, streak as 0, index i = 0.

While i is within bounds:

If nums[i] does not match curr, reset:

curr = nums[i]
streak = 0

Skip over all duplicates of curr by advancing i while nums[i] == curr.

Increase streak by 1 since we found the expected number.

Increase curr by 1 to expect the next number in the sequence.

Update res with the maximum streak found so far.

Return res after scanning the entire list.

In [ ]:
# Sorting
# Time Complexity: O(nlogn)
# Space Complexity: O(n)
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        if not nums:
            return 0
        res = 0
        nums.sort()

        curr, streak = nums[0], 0
        i = 0
        while i < len(nums):
            if curr != nums[i]:
                curr = nums[i]
                streak = 0
            while i < len(nums) and nums[i] == curr:
                i += 1
            streak += 1
            curr += 1
            res = max(res, streak)
        return res

# Hash Set

Intuition

To avoid repeatedly recounting the same sequences, we only want to start counting when we find the beginning of a consecutive sequence.

A number is the start of a sequence if num - 1 is not in the set.

This guarantees that each consecutive sequence is counted exactly once.

Once we identify such a starting number, we simply keep checking if num + 1, num + 2, … exist in the set and extend the streak as far as possible.

This makes the solution efficient and clean because each number contributes to the sequence only one time.

Algorithm

Convert the list into a set numSet for O(1) lookups.

Initialize longest to track the length of the longest consecutive sequence.

For each number num in numSet:

Check if num - 1 is not in the set:

If true, num is the start of a sequence.

Initialize length = 1.

While num + length exists in the set, increase length.

Update longest with the maximum length found.

Return longest after scanning all numbers.

In [ ]:
# Hash Set
# Time Complexity: O(n)
# Space Complexity: O(n)
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        numSet = set(nums)
        longest = 0

        for num in numSet:
            if (num - 1) not in numSet:
                length = 1
                while (num + length) in numSet:
                    length += 1
                longest = max(length, longest)
        return longest

# Hash Map

Intuition

When we place a new number into the map, it may connect two existing sequences or extend one of them.

Instead of scanning forward or backward, we only look at the lengths stored at the neighbors:

mp[num - 1] gives the length of the sequence ending right before num

mp[num + 1] gives the length of the sequence starting right after num

By adding these together and including the current number, we know the total length of the new merged sequence.

We then update the left boundary and right boundary of this sequence so the correct length can be retrieved later.

This keeps the whole operation very efficient and avoids repeated work.

Algorithm

Create a hash map mp that stores sequence lengths at boundary positions.

Initialize res = 0 to store the longest sequence found.

For each number num in the input:

If num is already in mp, skip it.

Compute the new sequence length:

length = mp[num - 1] + mp[num + 1] + 1

Store this length at num.

Update the boundaries:

Left boundary: mp[num - mp[num - 1]] = length

Right boundary: mp[num + mp[num + 1]] = length

Update res to keep track of the longest sequence.

Return res after processing all numbers.

In [ ]:
# Hash Map
# Time Complexity: O(n)
# Space Complexity: O(n)
from collections import defaultdict

class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        mp = defaultdict(int)
        res = 0

        for num in nums:
            if not mp[num]: # only process num if it hasn’t been seen before (mp[num] == 0)
                # mp[num - 1] → length of consecutive sequence ending just before num
                # mp[num + 1] → length of consecutive sequence starting just after num
                # + 1 → include num itself
                mp[num] = mp[num - 1] + mp[num + 1] + 1
                mp[num - mp[num - 1]] = mp[num]
                mp[num + mp[num + 1]] = mp[num]
                res = max(res, mp[num])
        return res